# PPEM Application Profiling Vizualizer 

Author: K. Desnos & MAI-Code-1.1-Flash  
Version : 1.0 (Sep. 2026)

## 0. Profile the application
1. Make sure the `ENABLE_PROFILING` is enabled for your C project.   
   In VSCode, press `[F1]`, search for `CMake: Edit CMake Cache (UI)`, and set the `ENABLE_PROFILING` to true.
2. Clean, Build and Run the executable in debug or release config.
3. Run the executable, a `trace.out` file should be generated in the execution directory. 

## 1. Configuration & trace parsing
In the following cell, indicate the location of the trace file and the associated executable.
The given files will be parsed to extract profiling info.

In [25]:
trace_file = "../build/Release/trace.out"  # Replace with the actual path to your trace file
executable = "../build/Release/mandelbrot.exe"  # Replace with the actual path to your executable

# Parse the given files
import trace_parser
calls, cores = trace_parser.parse_trace(trace_file, executable)

# Print number of function calls
print(f"Total function calls: {len(calls)}")

Total function calls: 4550


## 2. Analyze profiling data

### 2.1. Call Tree
This first array displays the call tree extracted from the graph. Each function call appears in the hierarchy of function call, with its given statistics.

- `call_count`: number of times this function was executed.
- `total_cpu_time_including_children`: cumulative execution time for all calls of this function, including the time spent in callees.
- `total_cpu_time_excluding_children`: cumulative execution time spent directly in this function body, excluding callees.
- `avg_cpu_time_including_children`: average duration per call, including callee time. <!-- `total_cpu_time_including_children` / `call_count`.!-->
- `avg_cpu_time_excluding_children`: average duration per call, excluding callee time. <!-- `total_cpu_time_excluding_children` / `call_count`. -->
- `percent_of_parent_incl`: how much of the parent function’s inclusive time is due to this function. <!-- `total_cpu_time_including_children` / `parent_total_cpu_time_including_children` * 100. -->
- `percent_of_parent_excl`: how much of the parent function’s exclusive time is due to this function. <!-- `total_cpu_time_excluding_children` / `parent_total_cpu_time_including_children` * 100. -->
- `percent_of_total_incl`: percentage of the total CPU time attributed to this function, including the time spent in callees.
- `percent_of_total_excl`: percentage of the total CPU time attributed to this function, excluding the time spent in callees.


These values help identify which functions or call branches dominate total runtime and which parts are mainly overhead versus actual work.

In [26]:
# Build a call tree from the parsed calls
call_tree_roots = trace_parser.analyze_call_tree(calls)
call_tree_df = trace_parser.build_call_tree_dataframe(call_tree_roots)

# Choose the columns you want to display, and set the order here.
# Edit this list to keep only the metrics you care about.
selected_columns = [
    "call_count",
    "percent_of_total_incl",
    "percent_of_total_excl",
    "total_cpu_time_including_children",
    "total_cpu_time_excluding_children",
    "avg_cpu_time_including_children",
    "avg_cpu_time_excluding_children",
    "percent_of_parent_incl",
    "percent_of_parent_excl"
]

# Keep only the existing columns, in the desired order.
visible_columns = [col for col in selected_columns if col in call_tree_df.columns]
call_tree_df = call_tree_df[visible_columns]

call_tree_df

,call_count,percent_of_total_incl,percent_of_total_excl,total_cpu_time_including_children,total_cpu_time_excluding_children,avg_cpu_time_including_children,avg_cpu_time_excluding_children,percent_of_parent_incl,percent_of_parent_excl
display_name,,,,,,,,,
main,1,100.00,1.02,19953427,204504,19953427.00,204504.00,100.00,1.02
├── yuvDisplayInit,1,0.61,0.61,121232,121218,121232.00,121218.00,0.61,0.61
│ └── startTiming,4,0.00,0.00,14,14,3.50,3.50,0.01,0.01
├── startTiming,7,0.00,0.00,15,15,2.14,2.14,0.00,0.00
├── mandelbrotToYUV,411,68.23,68.23,13614544,13614544,33125.41,33125.41,68.23,68.23
├── MD5_Update,411,16.23,0.10,3237650,19324,7877.49,47.02,16.23,0.10
│ ├── MD5_Init,411,0.00,0.00,977,977,2.38,2.38,0.03,0.03
│ ├── body,411,16.11,16.11,3215490,3215490,7823.58,7823.58,99.32,99.32
│ └── MD5_Final,411,0.01,0.01,1859,1196,4.52,2.91,0.06,0.04


### 2.1 Flame Graph view of the function call tree

This flame graph represents the inclusive CPU time of each function in the call tree. The wider a block, the more time it spent including its descendants.

In [27]:
# Plot the call hierarchy as an inclusive-time flame graph
flame_graph = trace_parser.build_flamegraph(calls)
flame_graph

## 3 Per function statistics

### 3.1. Function statistics
This array gathers all statistics for each encountered function call such that a function calls encountered in several places in the profiles program are summed.

In [28]:
# Aggregate all occurrences of the same function across the whole profile.
# This sums the values of repeated calls encountered at different places in the program.
function_stats_df = trace_parser.build_function_statistics_dataframe(calls)

# Choose the columns you want to display, and set the order here.
# Edit this list to keep only the metrics you care about.
selected_columns = [
    "call_count",
    "percent_of_total_incl",
    "percent_of_total_excl",
    "total_cpu_time_including_children",
    "total_cpu_time_excluding_children",
    "avg_cpu_time_including_children",
    "avg_cpu_time_excluding_children",
    "function_file",
    "function_address"
]

# Keep only the existing columns, in the desired order.
visible_columns = [col for col in selected_columns if col in function_stats_df.columns]
function_stats_df = function_stats_df[visible_columns]

function_stats_df

,call_count,percent_of_total_incl,percent_of_total_excl,total_cpu_time_including_children,total_cpu_time_excluding_children,avg_cpu_time_including_children,avg_cpu_time_excluding_children,function_file,function_address
function_name,,,,,,,,,
main,1,100.00,1.02,19953427,204504,19953427.00,204504.00,C:/Users/HP/Documents/PPEM/proj/projet_ppem-ma...,0000000140004A50
yuvDisplayInit,1,0.61,0.61,121232,121218,121232.00,121218.00,C:/Users/HP/Documents/PPEM/proj/projet_ppem-ma...,00000001400034A0
startTiming,422,0.00,0.00,609,609,1.44,1.44,C:/Users/HP/Documents/PPEM/proj/projet_ppem-ma...,00000001400014E0
mandelbrotToYUV,411,68.23,68.23,13614544,13614544,33125.41,33125.41,C:/Users/HP/Documents/PPEM/proj/projet_ppem-ma...,0000000140001920
MD5_Update,411,16.23,0.10,3237650,19324,7877.49,47.02,C:/Users/HP/Documents/PPEM/proj/projet_ppem-ma...,0000000140002C30
MD5_Init,411,0.00,0.00,977,977,2.38,2.38,C:/Users/HP/Documents/PPEM/proj/projet_ppem-ma...,0000000140002150
body,822,16.12,16.12,3216153,3216153,3912.59,3912.59,C:/Users/HP/Documents/PPEM/proj/projet_ppem-ma...,00000001400021A0
MD5_Final,411,0.01,0.01,1859,1196,4.52,2.91,C:/Users/HP/Documents/PPEM/proj/projet_ppem-ma...,0000000140002960
mandelbrotToYUV,411,11.10,11.10,2214188,2214188,5387.32,5387.32,C:/Users/HP/Documents/PPEM/proj/projet_ppem-ma...,0000000140001AB0


### 3.2 Function execution time distribution

The following chart allows to visualize the distribution of execution time for each function. 

In [29]:
# Select a subset of functions to display in the box plot.
# Use the function names exactly as they appear in the trace/profile table.
# Set this to None to display all functions.
selected_boxplot_functions = [
    "thread_wrapper",
    "yuvDisplay",
    "body"
]

# Keep only the selected functions; if a name is missing from the trace, it is ignored.
box_plot = trace_parser.build_boxplot(calls, selected_boxplot_functions)
box_plot